# 06 - SQL analysis

**Goal:** answer the same business questions in SQL, then check that the SQL results match the pandas results from notebooks 02-05.

**How it works**
1. Load `data/processed/transactions_clean.csv` into a SQLite database (`data/processed/retail.db`) using the schema in `sql/01_create_tables.sql`.
2. Run the scripts in `sql/` (02 views, 03 KPIs, 04 cohorts, 05 RFM, 06 churn). Each query starts with `-- name: <id>` so it can be run and displayed on its own.
3. Reconcile with the pandas outputs.

The `.sql` files run unchanged in any SQLite client (for example DB Browser for SQLite) against `retail.db`.

In [1]:
import sqlite3
import re
import time
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parent  # notebook runs from notebooks/
PROCESSED_DIR = ROOT / "data" / "processed"
SQL_DIR = ROOT / "sql"
DB_PATH = PROCESSED_DIR / "retail.db"

pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

## 1. Load the database

In [2]:
DB_PATH.unlink(missing_ok=True)
con = sqlite3.connect(DB_PATH)
con.executescript((SQL_DIR / "01_create_tables.sql").read_text(encoding="utf-8"))

started = time.time()
tx = pd.read_csv(PROCESSED_DIR / "transactions_clean.csv", dtype={"invoice": str, "stock_code": str})
for flag in ("is_cancellation", "is_product", "has_customer"):
    tx[flag] = tx[flag].astype(int)
tx["customer_id"] = tx.customer_id.astype("float")  # NaN -> NULL; SQLite's INTEGER affinity stores whole numbers as integers
tx.to_sql("transactions", con, if_exists="append", index=False, chunksize=100_000)
con.commit()

print(f"loaded {con.execute('SELECT COUNT(*) FROM transactions').fetchone()[0]:,} rows in {time.time() - started:.0f}s")
print(f"database size: {DB_PATH.stat().st_size / 1e6:.0f} MB")
del tx

loaded 1,027,016 rows in 16s
database size: 200 MB


## 2. Run the SQL scripts

`run_script` executes everything before the first `-- name:` marker (views, setup) and returns each named query as a DataFrame.

In [3]:
def run_script(filename):
    text = (SQL_DIR / filename).read_text(encoding="utf-8")
    parts = re.split(r"(?m)^-- name: ", text)  # markers at the start of a line only
    con.executescript(parts[0])                      # views / setup
    results = {}
    for block in parts[1:]:
        name, _, sql = block.partition("\n")
        results[name.strip()] = pd.read_sql_query(sql, con)
    return results

con.executescript((SQL_DIR / "02_views.sql").read_text(encoding="utf-8"))
kpis = run_script("03_kpis.sql")
cohorts = run_script("04_cohort_retention.sql")
rfm = run_script("05_rfm_segmentation.sql")
churn = run_script("06_churn_and_leakage.sql")
{name: len(frame) for group in (kpis, cohorts, rfm, churn) for name, frame in group.items()}

{'monthly_kpis': 25,
 'yoy_jan_nov': 2,
 'leakage_summary': 1,
 'customer_concentration': 4,
 'cohort_retention': 300,
 'repeat_rate_90d': 5,
 'rfm_segments': 10,
 'churn_overall': 1,
 'churn_by_recency_band': 6,
 'rule_silent_90_days': 1,
 'at_risk_now': 1,
 'at_risk_top_customers': 10}

### 2.1 KPIs (`03_kpis.sql`)

In [4]:
kpis["monthly_kpis"].tail(6)

,invoice_month,gross_sales,cancelled,cancel_rate,invoices,customers,avg_order_value
19,2011-07-01,"718,076.1200","37,919.1300",0.0528,1475,949,486.8300
20,2011-08-01,"757,841.3800","54,330.8000",0.0717,1361,935,556.8300
21,2011-09-01,"1,056,435.1900","38,838.5100",0.0368,1837,1266,575.0900
22,2011-10-01,"1,151,263.7300","81,895.5000",0.0711,2040,1364,564.3400
23,2011-11-01,"1,503,866.7800","47,720.9800",0.0317,2769,1664,543.1100
24,2011-12-01,"637,808.3300","205,089.2700",0.3216,819,615,778.7600


In [5]:
kpis["yoy_jan_nov"]

,year,gross_sales,growth_vs_prior_year
0,2010,"9,011,648.0000",NaN
1,2011,"9,182,868.0000",0.0190


In [6]:
kpis["leakage_summary"]

,gross_sales,cancelled_value,cancel_rate,guest_share_of_sales
0,"20,476,260.0000","1,462,424.0000",0.0714,0.1515


In [7]:
kpis["customer_concentration"]

,label,customers,share_of_sales
0,top 1%,58,0.3179
1,top 5%,293,0.5195
2,top 10%,587,0.6390
3,top 20%,1175,0.7723


### 2.2 Cohorts (`04_cohort_retention.sql`)

In [8]:
pivot = cohorts["cohort_retention"].pivot(index="cohort", columns="k", values="retention")
pivot.iloc[:6, :7]

k,0,1,2,3,4,5,6
cohort,,,,,,,
2009-12-01,1.0000,0.3529,0.3340,0.4251,0.3801,0.3592,0.3770
2010-01-01,1.0000,0.2063,0.3107,0.3055,0.2637,0.3003,0.2585
2010-02-01,1.0000,0.2380,0.2246,0.2914,0.2460,0.2005,0.1925
2010-03-01,1.0000,0.1896,0.2302,0.2415,0.2325,0.2032,0.2460
2010-04-01,1.0000,0.1939,0.1939,0.1633,0.1837,0.2245,0.2755
2010-05-01,1.0000,0.1575,0.1693,0.1732,0.1772,0.2559,0.2126


In [9]:
cohorts["repeat_rate_90d"]

,acquisition_half,customers,repeat_rate_90d
0,2009 H2,955,0.6147
1,2010 H1,2018,0.4529
2,2010 H2,1369,0.4098
3,2011 H1,699,0.4149
4,2011 H2,240,0.5583


### 2.3 RFM (`05_rfm_segmentation.sql`)

In [10]:
rfm["rfm_segments"]

,segment,customers,customer_share,revenue,revenue_share,avg_recency_days,avg_orders,avg_net_sales
0,Champions,1397,0.2393,"11,441,467.0000",0.6984,19.1000,16.2900,"8,190.0000"
1,Loyal,472,0.0809,"1,634,525.0000",0.0998,102.0000,9.1300,"3,463.0000"
2,At Risk,494,0.0846,"970,714.0000",0.0593,297.0000,5.6500,"1,965.0000"
3,Potential Loyalists,705,0.1208,"672,765.0000",0.0411,24.7000,2.8800,954.0000
4,Need Attention,695,0.1191,"549,616.0000",0.0335,107.7000,2.2500,791.0000
5,Lost,949,0.1626,"384,867.0000",0.0235,553.4000,1.2500,406.0000
6,About to Sleep,674,0.1155,"291,886.0000",0.0178,313.6000,1.3900,433.0000
7,Can't Lose Them,67,0.0115,"192,344.0000",0.0117,489.0000,8.6100,"2,871.0000"
8,Hibernating,152,0.0260,"165,904.0000",0.0101,494.4000,3.3900,"1,091.0000"
9,New Customers,232,0.0397,"78,540.0000",0.0048,29.2000,1.0000,339.0000


### 2.4 Churn (`06_churn_and_leakage.sql`)

In [11]:
churn["churn_overall"]

,base_customers,churn_rate
0,4326,0.4387


In [12]:
churn["churn_by_recency_band"]

,days_silent,customers,churn_rate
0,0-30,1016,0.1860
1,31-60,621,0.2528
2,61-90,354,0.3870
3,91-120,341,0.4692
4,121-180,392,0.5357
5,181-365,1602,0.6523


In [13]:
churn["rule_silent_90_days"]

,flagged,precision,recall
0,2335,0.6060,0.7455


In [14]:
churn["at_risk_now"]

,active_customers,flagged_customers,flagged_trailing_rev
0,4264,1365,"808,607.0000"


In [15]:
churn["at_risk_top_customers"]

,customer_id,days_silent,invoices,trailing_rev
0,15749,236,3,"21,536.0000"
1,16180,101,8,"10,217.0000"
2,12590,212,2,"9,861.0000"
3,12980,158,9,"7,092.0000"
4,13093,276,7,"6,942.0000"
5,13027,114,6,"6,912.0000"
6,16553,164,12,"5,669.0000"
7,12688,114,1,"4,874.0000"
8,14016,162,4,"4,328.0000"
9,12378,130,1,"4,009.0000"


## 3. Reconcile SQL against pandas

Every check compares a SQL result with the corresponding value from the pandas notebooks. Small differences are expected only where tie-breaking in quintile scoring differs.

In [16]:
checks = []

def check(name, sql_value, py_value, tol=1e-6):
    ok = bool(np.isclose(sql_value, py_value, rtol=0, atol=tol))
    checks.append({"check": name, "sql": sql_value, "pandas": py_value, "match": ok})

# KPIs
csv = pd.read_csv(PROCESSED_DIR / "transactions_clean.csv", usecols=["is_cancellation", "line_total", "has_customer"])
gross = csv.loc[~csv.is_cancellation, "line_total"].sum()
leak = kpis["leakage_summary"].iloc[0]
check("gross sales (GBP)", leak.gross_sales, round(gross, 0), tol=1)
check("cancelled value (GBP)", leak.cancelled_value, round(-csv.loc[csv.is_cancellation, "line_total"].sum(), 0), tol=1)
check("monthly gross sales sums to total", kpis["monthly_kpis"].gross_sales.sum(), gross, tol=1)
yoy = kpis["yoy_jan_nov"].set_index("year").growth_vs_prior_year
check("Jan-Nov 2011 growth", yoy["2011"], 0.0190, tol=0.0006)

conc = kpis["customer_concentration"].set_index("label").share_of_sales
check("top 10% customers' share of sales", conc["top 10%"], 0.639, tol=0.001)
check("top 1% customers' share of sales", conc["top 1%"], 0.318, tol=0.001)

# Cohorts: compare every cell present in both
py_cohort = pd.read_csv(PROCESSED_DIR / "cohort_retention.csv", parse_dates=["cohort"])
py_cohort["cohort"] = py_cohort.cohort.dt.strftime("%Y-%m-%d")
merged = cohorts["cohort_retention"].merge(py_cohort, on=["cohort", "k"], suffixes=("_sql", "_py"))
check("cohort cells compared", len(merged), len(cohorts["cohort_retention"]), tol=0)
check("max abs difference in cohort retention", merged.retention_sql.sub(merged.retention_py).abs().max(), 0.0, tol=0.0001)

# RFM
py_seg = pd.read_csv(PROCESSED_DIR / "rfm_segments.csv").set_index("segment")
sql_seg = rfm["rfm_segments"].set_index("segment")
check("RFM customers scored", sql_seg.customers.sum(), py_seg.customers.sum(), tol=0)
check("max abs diff in customers per segment", (sql_seg.customers - py_seg.customers).abs().max(), 0, tol=5)
check("Champions revenue share", sql_seg.loc["Champions", "revenue_share"], py_seg.loc["Champions", "revenue_share"], tol=0.005)

# Churn
bt = pd.read_csv(PROCESSED_DIR / "churn_rule_backtest.csv").set_index("rule")
rule = churn["rule_silent_90_days"].iloc[0]
check("churn base customers", churn["churn_overall"].base_customers[0], 4326, tol=0)
check("90-day rule: flagged", rule.flagged, bt.loc["silent > 90 days", "flagged"], tol=0)
check("90-day rule: precision", rule.precision, bt.loc["silent > 90 days", "precision"], tol=0.0005)
check("90-day rule: recall", rule.recall, bt.loc["silent > 90 days", "recall"], tol=0.0005)

now = pd.read_csv(PROCESSED_DIR / "at_risk_customers.csv")
now_flagged = now[now.flag_90d & (now.trailing_rev > 0)]
live = churn["at_risk_now"].iloc[0]
check("live list: active customers", live.active_customers, len(now), tol=0)
check("live list: flagged customers", live.flagged_customers, len(now_flagged), tol=0)
check("live list: flagged trailing revenue", live.flagged_trailing_rev, round(now_flagged.trailing_rev.sum(), 0), tol=1)

report = pd.DataFrame(checks)
report

,check,sql,pandas,match
0,gross sales (GBP),"20,476,260.0000","20,476,260.0000",True
1,cancelled value (GBP),"1,462,424.0000","1,462,424.0000",True
2,monthly gross sales sums to total,"20,476,260.4300","20,476,260.4300",True
3,Jan-Nov 2011 growth,0.0190,0.0190,True
4,top 10% customers' share of sales,0.6390,0.6390,True
5,top 1% customers' share of sales,0.3179,0.3180,True
6,cohort cells compared,300.0000,300.0000,True
7,max abs difference in cohort retention,0.0000,0.0000,True
8,RFM customers scored,"5,837.0000","5,837.0000",True
9,max abs diff in customers per segment,1.0000,0.0000,True


In [17]:
assert report.match.all(), report[~report.match]
print(f"all {len(report)} reconciliation checks passed")

all 18 reconciliation checks passed


## 4. Key takeaways

- The SQL scripts reproduce the pandas results: revenue, cancellations, concentration, cohort retention, RFM segments, the 90-day rule back-test and the live at-risk list.
- Techniques used: CTEs, window functions (`LAG`, `NTILE`, `ROW_NUMBER`, running totals), conditional aggregation, correlated `EXISTS` for the churn label, and views to share logic between queries.
- Quintile scores can differ by a handful of customers between SQL and pandas because tied values are ordered differently at bin edges, which is why the RFM segment counts have a small tolerance.

In [18]:
con.close()